# Módulo 2 — Manipulação de Dados com Pandas
### Curso Introdutório de Python para Ciência de Dados
**Disciplina:** T326 - Ciência de Dados | UNIFOR  
**Dataset:** Brazilian Cities — Dados socioeconômicos de 5.578 municípios brasileiros

---

## 📚 O que você vai aprender neste módulo

| Seção | Conteúdo |
|-------|----------|
| 2.1 | Introdução ao Pandas — Series e DataFrames |
| 2.2 | Leitura e inspeção do dataset |
| 2.3 | Limpeza e tratamento de dados |
| 2.4 | Manipulação básica e estatística descritiva |
| 2.5 | Exercícios práticos |


> 💡 **Dica:** Execute cada célula em sequência com `Shift + Enter`


---
## 2.1 Teoria — Introdução ao Pandas

**Pandas** é a principal biblioteca Python para análise de dados. Ela fornece duas estruturas fundamentais:

- **`Series`** → vetor unidimensional (como uma coluna de planilha)
- **`DataFrame`** → tabela bidimensional (como uma planilha completa)

```
DataFrame
┌──────────────┬─────────┬─────────┐
│     CITY     │  STATE  │  IDHM   │  ← colunas (Series)
├──────────────┼─────────┼─────────┤
│  São Paulo   │   SP    │  0.805  │  ← linha (índice 0)
│  Fortaleza   │   CE    │  0.754  │  ← linha (índice 1)
│  ...         │   ...   │  ...    │
└──────────────┴─────────┴─────────┘
```


In [ ]:
# ── Importações necessárias para todo o módulo ──────────────────
import pandas as pd      # Manipulação de dados
import numpy as np       # Operações numéricas
import warnings
warnings.filterwarnings('ignore')

print(f"✅ Pandas versão: {pd.__version__}")
print(f"✅ NumPy  versão: {np.__version__}")


✅ Pandas versão: 3.0.2
✅ NumPy  versão: 2.4.4


### Prática — Criando Series e DataFrames do zero

In [ ]:
# ── Criando uma Series ──────────────────────────────────────────
# Uma Series é basicamente uma lista com índice (rótulo)
populacao = pd.Series(
    [12_325_232, 2_703_391, 1_440_000, 3_849_322],
    index=['São Paulo', 'Fortaleza', 'Manaus', 'Salvador'],
    name='Populacao'
)

print("Series — Populações:")
print(populacao)
print(f"\nTipo: {type(populacao)}")
print(f"Dtype: {populacao.dtype}")


Series — Populações:
São Paulo    12325232
Fortaleza     2703391
Manaus        1440000
Salvador      3849322
Name: Populacao, dtype: int64

Tipo: <class 'pandas.Series'>
Dtype: int64


In [ ]:
# ── Acessando elementos de uma Series ───────────────────────────
print("Fortaleza:", populacao['Fortaleza'])         # pelo rótulo
print("Primeiro elemento:", populacao.iloc[0])       # pela posição
print("\nCidades com pop > 2 milhões:")
print(populacao[populacao > 2_000_000])


Fortaleza: 2703391
Primeiro elemento: 12325232

Cidades com pop > 2 milhões:
São Paulo    12325232
Fortaleza     2703391
Salvador      3849322
Name: Populacao, dtype: int64


In [ ]:
# ── Criando um DataFrame ────────────────────────────────────────
# Um DataFrame é um conjunto de Series com o mesmo índice
dados_cidades = {
    'Cidade':     ['São Paulo', 'Fortaleza', 'Manaus', 'Salvador'],
    'Estado':     ['SP', 'CE', 'AM', 'BA'],
    'Populacao':  [12_325_232, 2_703_391, 1_440_000, 3_849_322],
    'IDHM':       [0.805, 0.754, 0.737, 0.759],
    'Capital':    [True, True, True, True]
}

df_exemplo = pd.DataFrame(dados_cidades)
print(df_exemplo)
print(f"\nFormato: {df_exemplo.shape}  →  ({df_exemplo.shape[0]} linhas × {df_exemplo.shape[1]} colunas)")


      Cidade Estado  Populacao   IDHM  Capital
0  São Paulo     SP   12325232  0.805     True
1  Fortaleza     CE    2703391  0.754     True
2     Manaus     AM    1440000  0.737     True
3   Salvador     BA    3849322  0.759     True

Formato: (4, 5)  →  (4 linhas × 5 colunas)


---
## 2.2 Teoria — Leitura e Inspeção do Dataset

Para carregar dados de um arquivo CSV usamos `pd.read_csv()`. Após o carregamento, os primeiros passos são sempre:
1. Verificar o tamanho (`shape`)
2. Ver as primeiras linhas (`head`)
3. Inspecionar tipos e nulos (`info`, `dtypes`, `isnull`)


In [ ]:
# ── Carregando o dataset Brazilian Cities ────────────────────────
# O arquivo CSV original tem um formato especial de aspas duplas.
# A função abaixo trata isso automaticamente.

import io

def carregar_brazilian_cities(caminho):
    """
    Lê o arquivo brazilian_city.csv que usa quoting especial
    (cada linha é envolta em aspas duplas com campos também aspas-duplos).
    Retorna um DataFrame limpo.
    """
    with open(caminho, 'r', encoding='utf-8', newline='') as f:
        raw = f.read()
    
    # Separar linhas pelo delimitador CRLF
    linhas = raw.split('\r\n')
    
    # Remover a aspas externa de cada linha e desescapar aspas internas
    def corrigir_linha(linha):
        if linha.startswith('"') and linha.endswith('"'):
            linha = linha[1:-1]
        return linha.replace('""', '"')
    
    linhas_corrigidas = [corrigir_linha(l) for l in linhas if l.strip()]
    conteudo = '\n'.join(linhas_corrigidas)
    
    return pd.read_csv(io.StringIO(conteudo))

# Carrega o dataset
df = carregar_brazilian_cities('../datasets/brazilian_city.csv')

print(f"✅ Dataset carregado com sucesso!")
print(f"   Municípios: {df.shape[0]:,}")
print(f"   Variáveis:  {df.shape[1]}")


✅ Dataset carregado com sucesso!
   Municípios: 5,578
   Variáveis:  81


In [ ]:
# ── Primeiras linhas do dataset ─────────────────────────────────
# head(n) mostra as n primeiras linhas (padrão n=5)
df.head()


,CITY,STATE,CAPITAL,IBGE_RES_POP,IBGE_RES_POP_BRAS,IBGE_RES_POP_ESTR,IBGE_DU,IBGE_DU_URBAN,IBGE_DU_RURAL,IBGE_POP,...,Pu_Bank,Pr_Assets,Pu_Assets,Cars,Motorcycles,Wheeled_tractor,UBER,MAC,WAL-MART,POST_OFFICES
0,Abadia De Goiás,GO,0,6876,6876,0,2137,1546,591,5300,...,0,0,0,2158,1246,0,0,0,0,1
1,Abadia Dos Dourados,MG,0,6704,6704,0,2328,1481,847,4154,...,0,0,0,2227,1142,0,0,0,0,1
2,Abadiânia,GO,0,15757,15609,148,4655,3233,1422,10656,...,1,33724584,67091904,2838,1426,0,0,0,0,3
3,Abaetetuba,PA,0,141100,141040,60,31061,19057,12004,82956,...,4,76181384,800078483,5277,25661,0,0,0,0,2
4,Abaeté,MG,0,22690,22690,0,7694,6667,1027,18464,...,2,44974716,371922572,6928,2953,0,0,0,0,4


In [ ]:
# ── Últimas linhas ──────────────────────────────────────────────
df.tail(3)


,CITY,STATE,CAPITAL,IBGE_RES_POP,IBGE_RES_POP_BRAS,IBGE_RES_POP_ESTR,IBGE_DU,IBGE_DU_URBAN,IBGE_DU_RURAL,IBGE_POP,...,Pu_Bank,Pr_Assets,Pu_Assets,Cars,Motorcycles,Wheeled_tractor,UBER,MAC,WAL-MART,POST_OFFICES
5575,Érico Cardoso,BA,0,10859,10859,0,2659,542,2117,1999,...,0,0,0,655,1020,0,0,0,0,1
5576,Óbidos,PA,0,49333,49324,9,11263,6068,5195,25295,...,3,0,184494811,938,4985,0,0,0,0,1
5577,Óleo,SP,0,2673,2673,0,911,595,316,1763,...,0,0,0,866,172,0,0,0,0,3


In [ ]:
# ── Amostra aleatória ───────────────────────────────────────────
# Útil para ver uma visão representativa do dataset
df.sample(5, random_state=42)


,CITY,STATE,CAPITAL,IBGE_RES_POP,IBGE_RES_POP_BRAS,IBGE_RES_POP_ESTR,IBGE_DU,IBGE_DU_URBAN,IBGE_DU_RURAL,IBGE_POP,...,Pu_Bank,Pr_Assets,Pu_Assets,Cars,Motorcycles,Wheeled_tractor,UBER,MAC,WAL-MART,POST_OFFICES
2633,Kaloré,PR,0,4506,4502,4,1522,1080,442,3169,...,0,0,0,1515,780,1,0,0,0,1
1550,Dois Irmãos Do Buriti,MS,0,10363,10329,34,2592,1494,1098,4686,...,1,0,54057946,1690,1387,0,0,0,0,2
724,Braço Do Trombudo,SC,0,3457,3457,0,1121,645,476,1894,...,1,0,33042752,1500,687,1,0,0,0,1
2788,Luís Antônio,SP,0,11286,11281,5,3210,3105,105,10864,...,2,136344406,90843179,4159,856,0,0,0,0,1
468,Barra De Santana,PB,0,8206,8206,0,2431,249,2182,886,...,0,0,0,468,536,0,0,0,0,1


In [ ]:
# ── Informações gerais do dataset ───────────────────────────────
# info() mostra: número de linhas, colunas, tipos de dados e uso de memória
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 5578 entries, 0 to 5577
Data columns (total 81 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CITY                    5578 non-null   str    
 1   STATE                   5578 non-null   str    
 2   CAPITAL                 5578 non-null   int64  
 3   IBGE_RES_POP            5578 non-null   int64  
 4   IBGE_RES_POP_BRAS       5578 non-null   int64  
 5   IBGE_RES_POP_ESTR       5578 non-null   int64  
 6   IBGE_DU                 5578 non-null   int64  
 7   IBGE_DU_URBAN           5578 non-null   int64  
 8   IBGE_DU_RURAL           5578 non-null   int64  
 9   IBGE_POP                5578 non-null   int64  
 10  IBGE_1                  5578 non-null   int64  
 11  IBGE_1-4                5578 non-null   int64  
 12  IBGE_5-9                5578 non-null   int64  
 13  IBGE_10-14              5578 non-null   int64  
 14  IBGE_15-59              5578 non-null   int64  
 15

In [ ]:
# ── Tipos de dados de cada coluna ───────────────────────────────
print("Tipos de dados:")
print(df.dtypes.value_counts())
print()

# Colunas do tipo texto (object)
cols_texto = df.select_dtypes(include='object').columns.tolist()
print(f"Colunas de texto ({len(cols_texto)}): {cols_texto}")


Tipos de dados:
int64      66
float64     9
str         6
Name: count, dtype: int64

Colunas de texto (6): ['CITY', 'STATE', 'REGIAO_TUR', 'CATEGORIA_TUR', 'RURAL_URBAN', 'GVA_MAIN']


In [ ]:
# ── Estatísticas descritivas ────────────────────────────────────
# describe() calcula: count, mean, std, min, quartis, max para numéricas
df.describe().round(2)


,CAPITAL,IBGE_RES_POP,IBGE_RES_POP_BRAS,IBGE_RES_POP_ESTR,IBGE_DU,IBGE_DU_URBAN,IBGE_DU_RURAL,IBGE_POP,IBGE_1,IBGE_1-4,...,Pu_Bank,Pr_Assets,Pu_Assets,Cars,Motorcycles,Wheeled_tractor,UBER,MAC,WAL-MART,POST_OFFICES
count,5578.00,5578.00,5578.00,5578.00,5578.00,5578.00,5578.00,5578.00,5578.00,5578.00,...,5578.00,5.578000e+03,5.578000e+03,5578.00,5578.00,5578.00,5578.00,5578.00,5578.00,5578.00
mean,0.01,34223.13,34145.73,77.40,10283.13,8842.32,1440.81,27552.70,382.67,1542.09,...,0.95,5.500436e+09,3.598663e+09,9839.79,4869.56,5.74,0.02,0.13,0.04,2.04
std,0.08,202882.88,201262.67,1793.79,64691.99,64285.75,1690.48,185746.81,2324.18,9242.62,...,1.07,2.775752e+11,1.164327e+11,91757.28,20916.73,55.30,0.15,2.15,0.53,4.38
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000000e+00,0.000000e+00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,5217.00,5214.00,0.00,1565.25,870.00,469.25,2784.25,38.00,158.00,...,0.00,0.000000e+00,0.000000e+00,599.00,588.00,0.00,0.00,0.00,0.00,1.00
50%,0.00,10926.50,10916.00,0.00,3167.00,1839.50,916.00,6156.00,92.00,376.00,...,1.00,0.000000e+00,2.319925e+07,1431.50,1280.00,0.00,0.00,0.00,0.00,1.00
75%,0.00,23409.00,23380.00,10.00,6722.50,4618.75,1811.75,15298.75,232.00,949.75,...,2.00,4.774853e+07,1.991191e+08,4084.00,3292.75,1.00,0.00,0.00,0.00,2.00
max,1.00,11253503.00,11133776.00,119727.00,3576148.00,3548433.00,33809.00,10463636.00,129464.00,514794.00,...,8.00,1.947077e+13,8.016164e+12,5740995.00,1134570.00,3236.00,1.00,130.00,26.00,225.00


In [ ]:
# ── Dicionário de dados: entendendo as colunas principais ───────
dicionario = {
    'CITY':                 'Nome do município',
    'STATE':                'Estado (UF)',
    'CAPITAL':              '1 = Capital estadual, 0 = Não capital',
    'IBGE_RES_POP':         'População residente (Censo IBGE)',
    'ESTIMATED_POP':        'População estimada (mais recente)',
    'IDHM':                 'Índice de Desenvolvimento Humano Municipal',
    'IDHM_Renda':           'IDH - componente Renda',
    'IDHM_Longevidade':     'IDH - componente Longevidade',
    'IDHM_Educacao':        'IDH - componente Educação',
    'GDP':                  'PIB municipal (R$ mil)',
    'GDP_CAPITA':           'PIB per capita (R$)',
    'GVA_AGROPEC':          'Valor Adicionado Bruto — Agropecuária',
    'GVA_INDUSTRY':         'Valor Adicionado Bruto — Indústria',
    'GVA_SERVICES':         'Valor Adicionado Bruto — Serviços',
    'AREA':                 'Área territorial (km²)',
    'LONG':                 'Longitude geográfica',
    'LAT':                  'Latitude geográfica',
    'ALT':                  'Altitude (metros)',
    'Cars':                 'Número de automóveis registrados',
    'Motorcycles':          'Número de motocicletas registradas',
    'HOTELS':               'Número de hotéis',
    'BEDS':                 'Número de leitos hoteleiros',
    'RURAL_URBAN':          'Classificação: Urbano / Rural / Rural Adjacente',
}

print(f"{'COLUNA':<25} DESCRIÇÃO")
print("-" * 65)
for col, desc in dicionario.items():
    print(f"{col:<25} {desc}")


COLUNA                    DESCRIÇÃO
-----------------------------------------------------------------
CITY                      Nome do município
STATE                     Estado (UF)
CAPITAL                   1 = Capital estadual, 0 = Não capital
IBGE_RES_POP              População residente (Censo IBGE)
ESTIMATED_POP             População estimada (mais recente)
IDHM                      Índice de Desenvolvimento Humano Municipal
IDHM_Renda                IDH - componente Renda
IDHM_Longevidade          IDH - componente Longevidade
IDHM_Educacao             IDH - componente Educação
GDP                       PIB municipal (R$ mil)
GDP_CAPITA                PIB per capita (R$)
GVA_AGROPEC               Valor Adicionado Bruto — Agropecuária
GVA_INDUSTRY              Valor Adicionado Bruto — Indústria
GVA_SERVICES              Valor Adicionado Bruto — Serviços
AREA                      Área territorial (km²)
LONG                      Longitude geográfica
LAT                       Latitu

---
## 2.3 Teoria — Limpeza e Tratamento de Dados

Na prática, dados reais sempre têm problemas. As etapas clássicas de limpeza são:

| Problema | Solução Pandas |
|----------|---------------|
| Valores ausentes (NaN) | `fillna()`, `dropna()` |
| Duplicatas | `drop_duplicates()` |
| Tipos errados | `astype()`, `pd.to_numeric()` |
| Nomes inconsistentes | `.str.strip()`, `.str.upper()` |
| Outliers extremos | filtros booleanos, `clip()` |


In [ ]:
# ── 2.3.1 Verificando valores ausentes ──────────────────────────
nulos = df.isnull().sum()
nulos_pct = (df.isnull().mean() * 100).round(2)

# Montar relatório de nulos
relatorio_nulos = pd.DataFrame({
    'Ausentes': nulos,
    'Percentual (%)': nulos_pct
})

# Mostrar apenas colunas com pelo menos 1 nulo
relatorio_nulos = relatorio_nulos[relatorio_nulos['Ausentes'] > 0].sort_values('Ausentes', ascending=False)

if relatorio_nulos.empty:
    print("✅ Nenhum valor ausente encontrado no dataset!")
else:
    print(f"⚠️  {len(relatorio_nulos)} coluna(s) com valores ausentes:")
    print(relatorio_nulos)


✅ Nenhum valor ausente encontrado no dataset!


In [ ]:
# ── 2.3.2 Verificando duplicatas ────────────────────────────────
n_duplicatas = df.duplicated().sum()
print(f"Linhas duplicadas: {n_duplicatas}")

# Verificar duplicatas apenas pelo nome da cidade e estado
n_dup_cidade = df.duplicated(subset=['CITY', 'STATE']).sum()
print(f"Cidades com nome duplicado: {n_dup_cidade}")

# Ver quais cidades aparecem mais de uma vez
if n_dup_cidade > 0:
    cidades_dup = df[df.duplicated(subset=['CITY', 'STATE'], keep=False)].sort_values('CITY')
    print("\nExemplos:")
    print(cidades_dup[['CITY', 'STATE']].head(10))


Linhas duplicadas: 2
Cidades com nome duplicado: 3

Exemplos:
             CITY STATE
373         Assis    SP
374         Assis    SP
3290  Nova Fátima    PR
3291  Nova Fátima    PR
3561       Paraty    RJ
3562       Paraty    RJ


In [ ]:
# ── 2.3.3 Criando uma cópia limpa do DataFrame ──────────────────
# Sempre trabalhe com uma cópia para preservar o original!
df_limpo = df.copy()

# Renomear colunas problemáticas para facilitar o uso
df_limpo = df_limpo.rename(columns={
    'IDHM Ranking 2010':    'IDHM_Ranking',
    'IBGE_CROP_PRODUCTION_$': 'IBGE_CROP_PROD',
    'WAL-MART':              'WALMART',
    'IBGE_1-4':              'IBGE_1a4',
    'IBGE_5-9':              'IBGE_5a9',
    'IBGE_10-14':            'IBGE_10a14',
    'IBGE_15-59':            'IBGE_15a59',
    'IBGE_60+':              'IBGE_60mais',
})

print("✅ Colunas renomeadas!")
print(f"Dataset limpo: {df_limpo.shape}")


✅ Colunas renomeadas!
Dataset limpo: (5578, 81)


In [ ]:
# ── 2.3.4 Verificando e corrigindo tipos de dados ───────────────
# Conferir colunas que deveriam ser numéricas
cols_numericas_esperadas = ['IDHM', 'GDP_CAPITA', 'AREA', 'ALT']

for col in cols_numericas_esperadas:
    print(f"{col:<20} → tipo atual: {df_limpo[col].dtype}")


IDHM                 → tipo atual: float64
GDP_CAPITA           → tipo atual: float64
AREA                 → tipo atual: float64
ALT                  → tipo atual: float64


In [ ]:
# ── 2.3.5 Tratando municípios com população zero ─────────────────
# Municípios com IBGE_RES_POP = 0 são dados inválidos
pop_zero = df_limpo[df_limpo['IBGE_RES_POP'] == 0]
print(f"Municípios com população IBGE = 0: {len(pop_zero)}")
if not pop_zero.empty:
    print(pop_zero[['CITY', 'STATE', 'IBGE_RES_POP', 'ESTIMATED_POP']].head())

# Usar população estimada onde a do IBGE é zero
df_limpo['POP_FINAL'] = df_limpo['IBGE_RES_POP'].where(
    df_limpo['IBGE_RES_POP'] > 0,   # condição: pop IBGE > 0 → usa IBGE
    other=df_limpo['ESTIMATED_POP']  # senão → usa estimada
)
print(f"\n✅ Coluna POP_FINAL criada com {(df_limpo['POP_FINAL'] > 0).sum()} municípios válidos")


Municípios com população IBGE = 0: 10
                   CITY STATE  IBGE_RES_POP  ESTIMATED_POP
436    Balneário Rincão    SC             0          12570
2666    Lagoa Dos Patos    RS             0              0
2673        Lagoa Mirim    RS             0              0
3079   Mojuí Dos Campos    PA             0          15982
3569  Paraíso Das Águas    MS             0           5455

✅ Coluna POP_FINAL criada com 5574 municípios válidos


In [ ]:
# ── 2.3.6 Criando coluna de Densidade Demográfica ───────────────
# Densidade = População / Área  (hab/km²)
# Evitar divisão por zero com mask
df_limpo['DENSIDADE'] = np.where(
    df_limpo['AREA'] > 0,
    (df_limpo['POP_FINAL'] / df_limpo['AREA']).round(2),
    np.nan
)

print("Densidade demográfica (hab/km²) — Top 10:")
print(df_limpo[['CITY', 'STATE', 'POP_FINAL', 'AREA', 'DENSIDADE']]
      .sort_values('DENSIDADE', ascending=False)
      .head(10)
      .to_string(index=False))


Densidade demográfica (hab/km²) — Top 10:
              CITY STATE  POP_FINAL     AREA  DENSIDADE
São João De Meriti    RJ     458673   35.216   13024.56
           Diadema    SP     386089   30.732   12563.09
   Taboão Da Serra    SP     244528   20.388   11993.72
       Carapicuíba    SP     369584   34.546   10698.32
            Osasco    SP     666740   64.954   10264.80
São Caetano Do Sul    SP     149263   15.331    9736.03
            Olinda    PE     377779   41.300    9147.19
         Nilópolis    RJ     157425   19.393    8117.62
         Fortaleza    CE    2452185  312.407    7849.33
         São Paulo    SP   11253503 1521.110    7398.22


---
## 2.4 Teoria — Manipulação Básica e Estatística Descritiva

As operações mais frequentes em Pandas:

| Operação | Método |
|----------|--------|
| Selecionar colunas | `df[['col1', 'col2']]` |
| Filtrar linhas | `df[df['col'] > valor]` |
| Ordenar | `df.sort_values('col')` |
| Agrupar | `df.groupby('col').agg(...)` |
| Criar colunas | `df['nova'] = ...` |
| Pivot table | `df.pivot_table(...)` |


In [ ]:
# ── 2.4.1 Seleção de colunas ────────────────────────────────────
# Selecionar apenas as colunas de interesse
colunas_chave = ['CITY', 'STATE', 'POP_FINAL', 'IDHM', 'GDP_CAPITA', 'AREA', 'DENSIDADE', 'RURAL_URBAN']
df_chave = df_limpo[colunas_chave].copy()

print(f"DataFrame reduzido: {df_chave.shape}")
df_chave.head()


DataFrame reduzido: (5578, 8)


,CITY,STATE,POP_FINAL,IDHM,GDP_CAPITA,AREA,DENSIDADE,RURAL_URBAN
0,Abadia De Goiás,GO,6876,0.708,20665.0,147.256,46.69,Urbano
1,Abadia Dos Dourados,MG,6704,0.689,25592.0,881.064,7.61,Rural Adjacente
2,Abadiânia,GO,15757,0.689,15628.0,1045.127,15.08,Rural Adjacente
3,Abaetetuba,PA,141100,0.628,8222.0,1610.651,87.60,Urbano
4,Abaeté,MG,22690,0.698,18250.0,1817.067,12.49,Urbano


In [ ]:
# ── 2.4.2 Filtragem com condições ───────────────────────────────
# Filtros simples
capitais = df_limpo[df_limpo['CAPITAL'] == 1]
print(f"Capitais estaduais: {len(capitais)}")

# Filtro composto — municípios com IDHM alto E grande população
grandes_e_ricos = df_limpo[
    (df_limpo['IDHM'] >= 0.750) &
    (df_limpo['POP_FINAL'] >= 100_000)
]
print(f"Municípios com IDHM ≥ 0.75 e pop ≥ 100 mil: {len(grandes_e_ricos)}")

# Filtro com isin — municípios da Região Norte
estados_norte = ['AM', 'PA', 'RR', 'RO', 'AC', 'AP', 'TO']
norte = df_limpo[df_limpo['STATE'].isin(estados_norte)]
print(f"Municípios da Região Norte: {len(norte)}")


Capitais estaduais: 33
Municípios com IDHM ≥ 0.75 e pop ≥ 100 mil: 146
Municípios da Região Norte: 450


In [ ]:
# ── 2.4.3 Ordenação ─────────────────────────────────────────────
# Top 10 municípios com maior PIB per capita
top_pib = (df_limpo[['CITY', 'STATE', 'GDP_CAPITA', 'POP_FINAL', 'IDHM']]
           .sort_values('GDP_CAPITA', ascending=False)
           .head(10)
           .reset_index(drop=True))

top_pib.index += 1  # ranking começa em 1
print("🏆 Top 10 — Maior PIB per Capita:")
print(top_pib.to_string())


🏆 Top 10 — Maior PIB per Capita:
                      CITY STATE  GDP_CAPITA  POP_FINAL   IDHM
1                 Paulínia    SP    314638.0      82146  0.795
2                 Selvíria    MS    306139.0       6287  0.682
3   São Francisco Do Conde    BA    296459.0      33183  0.674
4                  Triunfo    RS    289932.0      25793  0.733
5             Brejo Alegre    SP    274572.0       2573  0.710
6   Sebastianópolis Do Sul    SP    253147.0       3031  0.773
7                 Louveira    SP    250827.0      37125  0.777
8          Campos De Júlio    MT    202309.0       5154  0.744
9                Meridiano    SP    184603.0       3855  0.731
10                 Extrema    MG    183218.0      28599  0.732


In [ ]:
# ── 2.4.4 Criando categorias com cut e qcut ─────────────────────
# cut() — intervalos fixos definidos pelo analista
df_limpo['IDHM_FAIXA'] = pd.cut(
    df_limpo['IDHM'],
    bins=[0, 0.499, 0.599, 0.699, 0.799, 1.0],
    labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto']
)

contagem_faixas = df_limpo['IDHM_FAIXA'].value_counts().sort_index()
print("Municípios por faixa de IDHM:")
for faixa, n in contagem_faixas.items():
    barra = '█' * (n // 80)
    print(f"  {str(faixa):<12} {n:>5} {barra}")


Municípios por faixa de IDHM:
  Muito Baixo     32 
  Baixo         1367 █████████████████
  Médio         2235 ███████████████████████████
  Alto          1889 ███████████████████████
  Muito Alto      45 


In [ ]:
# ── 2.4.5 Agrupamentos com groupby ──────────────────────────────
# Estatísticas por Estado — usando agg() para múltiplas métricas
resumo_estado = df_limpo.groupby('STATE').agg(
    N_Municipios  = ('CITY',        'count'),
    Pop_Total     = ('POP_FINAL',   'sum'),
    IDHM_Medio   = ('IDHM',        'mean'),
    PIB_Medio    = ('GDP_CAPITA',  'mean'),
    Maior_Cidade  = ('POP_FINAL',   'max'),
).round(3)

# Ordenar por IDHM médio (descrescente)
resumo_estado = resumo_estado.sort_values('IDHM_Medio', ascending=False)
print("Resumo por Estado (Top 10 IDHM Médio):")
print(resumo_estado.head(10).to_string())


Resumo por Estado (Top 10 IDHM Médio):
       N_Municipios  Pop_Total  IDHM_Medio  PIB_Medio  Maior_Cidade
STATE                                                              
DF                1    2570160       0.824  79100.000       2570160
SP              646   41357343       0.740  32174.187      11253503
SC              295    6271028       0.727  30630.298        515288
RS              499   10696897       0.709  33590.038       1409351
RJ               93   16027462       0.709  30850.323       6320446
PR              400   10452673       0.702  29169.958       1751907
GO              246    6003788       0.695  26278.805       1302001
ES               78    3514952       0.692  21217.526        414586
MT              141    3035122       0.684  36405.057        551098
MS               79    2454479       0.671  37003.823        786797


In [ ]:
# ── 2.4.6 Pivot Table — análise cruzada ────────────────────────
# Cruzar Região (Urbano/Rural) × Faixa de IDHM
pivot = pd.pivot_table(
    df_limpo,
    values='CITY',
    index='RURAL_URBAN',
    columns='IDHM_FAIXA',
    aggfunc='count',
    fill_value=0
)

print("Tabela Pivô — Quantidade de Municípios por Zona × IDHM:")
print(pivot.to_string())


Tabela Pivô — Quantidade de Municípios por Zona × IDHM:
IDHM_FAIXA               Muito Baixo  Baixo  Médio  Alto  Muito Alto
RURAL_URBAN                                                         
0                                  0      2      0     0           0
Intermediário Adjacente            0    125    311   251           1
Intermediário Remoto               2     20     31     7           0
Rural Adjacente                   14    977   1354   692           1
Rural Remoto                      16    159    132    16           0
Urbano                             0     84    407   923          43


In [ ]:
# ── 2.4.7 Estatísticas descritivas por grupo ────────────────────
# Comparar IDH das capitais vs demais municípios
df_limpo['TIPO'] = df_limpo['CAPITAL'].map({1: 'Capital', 0: 'Interior'})

stats_tipo = df_limpo.groupby('TIPO')[['IDHM', 'GDP_CAPITA', 'POP_FINAL']].describe().round(3)
print("Estatísticas: Capitais vs Interior")
print(stats_tipo[['IDHM', 'GDP_CAPITA']].to_string())


Estatísticas: Capitais vs Interior
            IDHM                                                  GDP_CAPITA                                                                  
           count   mean    std    min    25%    50%    75%    max      count       mean       std     min      25%      50%      75%       max
TIPO                                                                                                                                          
Capital     33.0  0.748  0.073  0.524  0.733  0.763  0.799  0.847       33.0  30009.515  15633.59  6824.0  20853.0  25718.0  35122.0   79100.0
Interior  5545.0  0.657  0.077  0.000  0.599  0.665  0.717  0.862     5545.0  21057.869  20342.70     0.0   9036.0  15789.0  26121.0  314638.0


In [ ]:
# ── 2.4.8 Correlação entre variáveis ────────────────────────────
# Quais variáveis se correlacionam com o IDHM?
variaveis_analise = ['IDHM', 'GDP_CAPITA', 'POP_FINAL', 'DENSIDADE', 'AREA',
                      'PAY_TV', 'FIXED_PHONES', 'Cars', 'Motorcycles']

corr = df_limpo[variaveis_analise].corr()['IDHM'].drop('IDHM').sort_values(ascending=False)

print("Correlação com IDHM:")
for var, val in corr.items():
    barra_pos = '▶' * int(abs(val) * 20) if val > 0 else ''
    barra_neg = '◀' * int(abs(val) * 20) if val < 0 else ''
    sinal = '+' if val > 0 else '-'
    print(f"  {var:<20} {sinal}{abs(val):.3f}  {barra_neg}{barra_pos}")


Correlação com IDHM:
  GDP_CAPITA           +0.471  ▶▶▶▶▶▶▶▶▶
  Motorcycles          +0.180  ▶▶▶
  DENSIDADE            +0.169  ▶▶▶
  Cars                 +0.138  ▶▶
  POP_FINAL            +0.132  ▶▶
  PAY_TV               +0.120  ▶▶
  FIXED_PHONES         +0.112  ▶▶
  AREA                 -0.125  ◀◀


---
## 2.5 Exercícios Práticos — Módulo 2

> Use o DataFrame `df_limpo` para responder às questões abaixo.  
> O **gabarito** está no arquivo `gabarito_exercicios.ipynb`.

---

### Exercício 1 — Filtragem básica
Liste todos os municípios do **Ceará (CE)** com IDHM **maior que 0.700**, ordenados do maior para o menor IDHM. Mostre as colunas: CITY, IDHM, GDP_CAPITA.

```python
# Seu código aqui
```

---

### Exercício 2 — Agrupamento
Calcule, por **Estado**, a **soma da população** e o **IDHM médio**. Quais são os 5 estados mais populosos?

```python
# Seu código aqui
```

---

### Exercício 3 — Criação de coluna
Crie uma coluna chamada `FROTA_TOTAL` somando `Cars` e `Motorcycles`. Depois, calcule a **frota média por habitante** em cada estado (frota_total / pop_final).

```python
# Seu código aqui
```

---

### Exercício 4 — Análise de capitais
Compare o **PIB per capita médio** das capitais estaduais com o das demais cidades. Qual a diferença percentual?

```python
# Seu código aqui
```

---

### Exercício 5 — Estatísticas descritivas
Calcule média, mediana, desvio padrão e quartis do IDHM para as cidades classificadas como **"Urbano"**, **"Rural Adjacente"** e **"Rural Remoto"** separadamente.

```python
# Seu código aqui
```


In [ ]:
# Espaço para suas respostas

# Exercício 1

# Exercício 2

# Exercício 3

# Exercício 4

# Exercício 5


---
## 📌 Resumo — Comandos Essenciais do Pandas

```python
# Leitura
df = pd.read_csv('arquivo.csv')

# Inspeção
df.shape          # (linhas, colunas)
df.head()         # primeiras 5 linhas
df.info()         # tipos + nulos
df.describe()     # estatísticas numéricas

# Seleção
df['coluna']                    # uma coluna (Series)
df[['col1', 'col2']]            # múltiplas colunas
df.iloc[0:5]                    # por posição
df.loc[df['col'] > valor]       # por condição

# Limpeza
df.isnull().sum()               # contar nulos
df.dropna()                     # remover nulos
df.fillna(0)                    # preencher nulos
df.drop_duplicates()            # remover duplicatas
df.rename(columns={'a': 'b'})   # renomear

# Transformação
df['nova'] = df['col'] * 2      # nova coluna
pd.cut(df['col'], bins=[...])   # categorizar
df.groupby('col').agg(...)      # agrupar
df.pivot_table(...)             # tabela pivô
df.sort_values('col')           # ordenar
```

---
*Próximo: **Módulo 3 — Visualização de Dados***
